In [1]:
# -----------------------------------------------
# IMPORTANT:  File used here should be the output
# result of data pre-processing
# -----------------------------------------------
# Import pandas library
import pandas as pd
import numpy as np
import dask.dataframe as dd
import dask.array as da

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

pd.options.display.float_format = '{:.2f}'.format

<h2>DATA PREPARATION</h2>

In [2]:
#   Dask infers data types as it reads csv files in small chunks
#   for efficiency, for some columns the assumed data types might
#   not be the correct ones, which is the case for 'preparation_state_code'
#   One way to handle the error generated by this issue is by 
#   setting the data types of the columns beforehand.
#   Usually, strings are declared as string[pyarrow], but in the case
#   of columns with only a few categories, the 'category' type is more
#   efficient which is the case for market country (US and NZ only).
dtype_dict = {
    'market_country': 'category',  
    'Category': 'category',  
}
df_food = dd.read_csv("categorized_ratios_final_output.csv", dtype=dtype_dict)
total_rows = len(df_food)


In [3]:
print('Number of rows in current pre-processed data set',total_rows)

# Check for missing values in a specific column (e.g., 'preparation_state_code')
missing_values_count = df_food.isnull().sum()

# Compute and print the number of missing values
print('Missing Values')
print(missing_values_count.compute())

Number of rows in current pre-processed data set 1177216
Missing Values
Unnamed: 0.1                  0
Unnamed: 0                    0
fdc_id                        0
brand_owner                9252
brand_name               338519
ingredients                1778
serving_size                  0
serving_size_unit          5587
branded_food_category      5449
market_country                0
Carbs                         0
Fiber                         0
Ratio                         0
Category                      0
Proteins                   4330
Total_Fat                 19034
Total_Sugars              25635
description                   0
dtype: int64


<h2>SUMMARY STATISTICS</h2>

In [4]:
# Summary statistics
# Similar to Pandas' df.describe()
summary = df_food.describe()
#   Not interested in summary of brand id column
summary = summary.drop(columns=['fdc_id'])
print(summary.compute())  # Call compute() only when you need the result

       Unnamed: 0.1  Unnamed: 0  serving_size      Carbs      Fiber  \
count    1177216.00  1177216.00    1177216.00 1177216.00 1177216.00   
mean       22119.41    82781.96        104.90      21.76       2.00   
std        12771.33    48102.19        100.45      17.51       2.84   
min            0.00        0.00          0.20       0.00       0.00   
25%        13647.00    42532.00         30.00       7.02       0.00   
50%        26371.00    84207.00         75.00      15.71       1.00   
75%        35439.00   125639.75        135.00      33.60       2.90   
max        44670.00   166850.00      10800.00      59.99      15.90   

           Ratio   Proteins  Total_Fat  Total_Sugars  
count 1177216.00 1172886.00 1158182.00    1151581.00  
mean       13.63       7.53      12.30          9.89  
std        13.73       9.75      15.33         12.13  
min         0.00       0.00       0.00          0.00  
25%         4.03       0.91       0.00          1.76  
50%         8.85       4.58   

In [5]:
# Check for missing values in a specific column (e.g., 'preparation_state_code')
missing_values_count = df_food.isnull().sum()

# Compute and print the number of missing values
print('Missing Values')
print(missing_values_count.compute())

Missing Values
Unnamed: 0.1                  0
Unnamed: 0                    0
fdc_id                        0
brand_owner                9252
brand_name               338519
ingredients                1778
serving_size                  0
serving_size_unit          5587
branded_food_category      5449
market_country                0
Carbs                         0
Fiber                         0
Ratio                         0
Category                      0
Proteins                   4330
Total_Fat                 19034
Total_Sugars              25635
description                   0
dtype: int64


In [6]:
df_food['brand_name'] = df_food['brand_name'].fillna("No Brand")
df_food['brand_owner'] = df_food['brand_owner'].fillna("Unknown Owner")
df_food['description'] = df_food['description'].fillna("No Description")
df_food['branded_food_category'] = df_food['branded_food_category'].fillna("No Food Category")
df_food['ingredients'] = df_food['ingredients'].fillna("Not Provided")
df_food['serving_size_unit'] = df_food['serving_size_unit'].fillna("Not Provided")

df_food['Proteins'] = df_food['Proteins'].fillna(0)
df_food['Total_Fat'] = df_food['Total_Fat'].fillna(0)
df_food['Total_Sugars'] = df_food['Total_Sugars'].fillna(0)



In [24]:
avoid_count = df_food[df_food['Category'] == 'Avoid'].shape[0]
best_count = df_food[df_food['Category'] == 'Best'].shape[0]
ideal_count = df_food[df_food['Category'] == 'Ideal'].shape[0]
moderate_count = df_food[df_food['Category'] == 'Moderate'].shape[0]

In [25]:
print("Best Count: ", best_count.compute())
print("Ideal Count: ", ideal_count.compute())
print("Moderate Count: ", moderate_count.compute())
print("Avoid Count: ", avoid_count.compute())

Best Count:  169212
Ideal Count:  198476
Moderate Count:  261787
Avoid Count:  547741


In [ ]:
import dask.array as da

In [56]:
# Select relevant columns (nutrients)
nutrient_columns = ['Carbs', 'Proteins', 'Total_Fat', 'Total_Sugars', 'Fiber']  # Modify based on your data

# Subset the DataFrame
data = df_food[nutrient_columns].fillna(0)

In [59]:
%pip install nltk

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   -------------------- ------------------- 0.8/1.5 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 4.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [7]:
df_food.compute()

,Unnamed: 0.1,Unnamed: 0,fdc_id,brand_owner,brand_name,ingredients,serving_size,serving_size_unit,branded_food_category,market_country,Carbs,Fiber,Ratio,Category,Proteins,Total_Fat,Total_Sugars,description
0,0,128424,2651496,"Wal-Mart Stores, Inc.",GREAT VALUE,"ORGANIC APPLES, WATER, ASCORBIC ACID (TO MAINT...",122.00,GRM,Wholesome Snacks,United States,11.48,0.80,14.35,Avoid,0.00,0.00,9.02,"UNSWEETENED ORGANIC APPLESAUCE, UNSWEETENED"
1,1,128497,2652255,Unknown Owner,BOWL & BASKET,"MILK, LIQUID CANE SUGAR (SUGAR, WATER), COCOA ...",473.00,MLT,Milk,United States,10.57,0.00,10.57,Avoid,3.38,3.81,10.57,"BOWL & BASKET CHOCOLATE MILK, 1 PINT"
2,2,128597,2653418,Seneca Foods Corporation,GREEN VALLEY,"ORGANIC PINTO BEANS, WATER, SEA SALT.",125.00,GRM,Canned & Bottled Beans,United States,15.20,4.00,3.80,Ideal,5.60,0.00,0.80,PINTO BEANS
3,3,128647,2653915,Unknown Owner,DAIZY FARMS,"BLACK BEANS, WATER, SALT, CALCIUM CHLORIDE (TO...",130.00,GRM,Canned & Bottled Beans,United States,16.92,6.90,2.45,Best,6.15,0.00,0.00,"BLACK BEANS, BLACK"
4,4,128720,2654810,Unknown Owner,SOL & MAR,"PITTED GREEN OLIVES, SUNFLOWER OIL, SALT, SPIC...",15.00,GRM,"Pickles, Olives, Peppers & Relishes",United States,6.67,6.70,1.00,Best,0.00,20.00,0.00,PITTED GREEN OLIVES WITH HOT CHILI PEPPER & SP...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107376,44666,128197,2649386,Seneca Foods Corp,STOKELY'S,"CARROTS, WATER",120.00,GRM,Vegetables Prepared/Processed,United States,3.33,0.80,4.16,Ideal,0.00,0.00,1.67,Stokely's No Salt Sliced Carrots 14.5oz
107377,44667,128210,2649517,East Side Entrees,ES Foods,Cinnamon Toast Crunch (25% Less Sugar): Whole ...,167.00,GRM,Processed Cereal Products,United States,32.93,3.00,10.98,Avoid,1.20,3.29,14.97,Breakfast Kit Cinnamon Toast Crunch Reduced Sugar
107378,44668,128280,2650192,Tyson Foods Inc.,Tyson,"Chicken wing sections, water, wheat flour, Con...",70.00,GRM,Meat/Poultry/Other Animals Prepared/Processed,United States,5.32,0.50,10.64,Avoid,21.17,0.00,0.24,Tyson Fire Stingers Fully Cooked Breaded Bone-...
107379,44669,128294,2650314,Kellogg Company US,Morningstar Farms,"INGREDIENTS: WATER, SOYBEAN OIL, MODIFIED CORN...",16.00,GRM,Vegetable Based Products / Meals,United States,12.60,0.50,25.20,Avoid,11.40,0.00,0.60,Morningstar Farms Breakfast Bacon 5.25oz


In [7]:
columns_to_remove = ['Unnamed: 0.1', 'Unnamed: 0']
df_food = df_food.drop(columns=columns_to_remove)

In [8]:
df_food['Ratio'] = df_food['Ratio'].round(2)

In [54]:
new_df = df_food.compute()
new_df.to_csv('categorized_ratios_output2.csv', index=True)

In [32]:
# Then save it to JSON
new_df.to_json('nutrition.json', orient='records', lines=False)

In [34]:
import json

with open("nutrition.json", "r") as f:
    data = json.load(f)

with open("data.ndjson", "w") as out:
    for item in data:
        out.write(json.dumps(item) + "\n")


In [17]:
df_food.compute()

,fdc_id,brand_owner,brand_name,ingredients,serving_size,serving_size_unit,branded_food_category,market_country,Carbs,Fiber,Ratio,Category,Proteins,Total_Fat,Total_Sugars,description,split_words
0,2651496,"Wal-Mart Stores, Inc.",GREAT VALUE,"ORGANIC APPLES, WATER, ASCORBIC ACID (TO MAINT...",122.00,GRM,Wholesome Snacks,United States,11.48,0.80,14.35,Avoid,0.00,0.00,9.02,"UNSWEETENED ORGANIC APPLESAUCE, UNSWEETENED",[Wholesome Snacks]
1,2652255,<NA>,BOWL & BASKET,"MILK, LIQUID CANE SUGAR (SUGAR, WATER), COCOA ...",473.00,MLT,Milk,United States,10.57,0.00,10.57,Avoid,3.38,3.81,10.57,"BOWL & BASKET CHOCOLATE MILK, 1 PINT",[Milk]
2,2653418,Seneca Foods Corporation,GREEN VALLEY,"ORGANIC PINTO BEANS, WATER, SEA SALT.",125.00,GRM,Canned & Bottled Beans,United States,15.20,4.00,3.80,Ideal,5.60,0.00,0.80,PINTO BEANS,[Canned & Bottled Beans]
3,2653915,<NA>,DAIZY FARMS,"BLACK BEANS, WATER, SALT, CALCIUM CHLORIDE (TO...",130.00,GRM,Canned & Bottled Beans,United States,16.92,6.90,2.45,Best,6.15,0.00,0.00,"BLACK BEANS, BLACK",[Canned & Bottled Beans]
4,2654810,<NA>,SOL & MAR,"PITTED GREEN OLIVES, SUNFLOWER OIL, SALT, SPIC...",15.00,GRM,"Pickles, Olives, Peppers & Relishes",United States,6.67,6.70,1.00,Best,0.00,20.00,0.00,PITTED GREEN OLIVES WITH HOT CHILI PEPPER & SP...,"[Pickles, Olives, Peppers & Relishes]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107376,2649386,Seneca Foods Corp,STOKELY'S,"CARROTS, WATER",120.00,GRM,Vegetables Prepared/Processed,United States,3.33,0.80,4.16,Ideal,0.00,NaN,1.67,Stokely's No Salt Sliced Carrots 14.5oz,[Vegetables Prepared/Processed]
107377,2649517,East Side Entrees,ES Foods,Cinnamon Toast Crunch (25% Less Sugar): Whole ...,167.00,GRM,Processed Cereal Products,United States,32.93,3.00,10.98,Avoid,1.20,3.29,14.97,Breakfast Kit Cinnamon Toast Crunch Reduced Sugar,[Processed Cereal Products]
107378,2650192,Tyson Foods Inc.,Tyson,"Chicken wing sections, water, wheat flour, Con...",70.00,GRM,Meat/Poultry/Other Animals Prepared/Processed,United States,5.32,0.50,10.64,Avoid,21.17,NaN,0.24,Tyson Fire Stingers Fully Cooked Breaded Bone-...,[Meat/Poultry/Other Animals Prepared/Processed]
107379,2650314,Kellogg Company US,Morningstar Farms,"INGREDIENTS: WATER, SOYBEAN OIL, MODIFIED CORN...",16.00,GRM,Vegetable Based Products / Meals,United States,12.60,0.50,25.20,Avoid,11.40,NaN,0.60,Morningstar Farms Breakfast Bacon 5.25oz,[Vegetable Based Products / Meals]


<h1>CLUSTERING ANALYSIS</h1>

In [ ]:
# Apply K-means clustering with Dask
kmeans = KMeans(n_clusters=3, random_state=42)

# Fit the model (Dask arrays will be handled by scikit-learn)
kmeans.fit(data_scaled)

# Assign cluster labels to the original DataFrame
clusters = kmeans.predict(data_scaled)

# Convert the predictions into a Dask DataFrame
cluster_predictions_dask = dd.from_array(clusters, columns=['Cluster'])



# Compute the results (Dask is lazy, so we need to compute it)
array_computed = cluster_predictions_dask.compute()



# Preview the DataFrame with cluster labels
#array_computed.head()


,Cluster
0,1
1,1
2,1
3,1
4,1


In [28]:
df_food['Ratio'] = df_food['Ratio'].round(2)

In [29]:
df_food.compute()

,fdc_id,brand_owner,brand_name,ingredients,serving_size,serving_size_unit,branded_food_category,market_country,Carbs,Fiber,Ratio,Category,Proteins,Total_Fat,Total_Sugars,description
0,2651496,"Wal-Mart Stores, Inc.",GREAT VALUE,"ORGANIC APPLES, WATER, ASCORBIC ACID (TO MAINT...",122.00,GRM,Wholesome Snacks,United States,11.48,0.80,14.35,Avoid,0.00,0.00,9.02,"UNSWEETENED ORGANIC APPLESAUCE, UNSWEETENED"
1,2652255,<NA>,BOWL & BASKET,"MILK, LIQUID CANE SUGAR (SUGAR, WATER), COCOA ...",473.00,MLT,Milk,United States,10.57,0.00,10.57,Avoid,3.38,3.81,10.57,"BOWL & BASKET CHOCOLATE MILK, 1 PINT"
2,2653418,Seneca Foods Corporation,GREEN VALLEY,"ORGANIC PINTO BEANS, WATER, SEA SALT.",125.00,GRM,Canned & Bottled Beans,United States,15.20,4.00,3.80,Ideal,5.60,0.00,0.80,PINTO BEANS
3,2653915,<NA>,DAIZY FARMS,"BLACK BEANS, WATER, SALT, CALCIUM CHLORIDE (TO...",130.00,GRM,Canned & Bottled Beans,United States,16.92,6.90,2.45,Best,6.15,0.00,0.00,"BLACK BEANS, BLACK"
4,2654810,<NA>,SOL & MAR,"PITTED GREEN OLIVES, SUNFLOWER OIL, SALT, SPIC...",15.00,GRM,"Pickles, Olives, Peppers & Relishes",United States,6.67,6.70,1.00,Best,0.00,20.00,0.00,PITTED GREEN OLIVES WITH HOT CHILI PEPPER & SP...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107376,2649386,Seneca Foods Corp,STOKELY'S,"CARROTS, WATER",120.00,GRM,Vegetables Prepared/Processed,United States,3.33,0.80,4.16,Ideal,0.00,NaN,1.67,Stokely's No Salt Sliced Carrots 14.5oz
107377,2649517,East Side Entrees,ES Foods,Cinnamon Toast Crunch (25% Less Sugar): Whole ...,167.00,GRM,Processed Cereal Products,United States,32.93,3.00,10.98,Avoid,1.20,3.29,14.97,Breakfast Kit Cinnamon Toast Crunch Reduced Sugar
107378,2650192,Tyson Foods Inc.,Tyson,"Chicken wing sections, water, wheat flour, Con...",70.00,GRM,Meat/Poultry/Other Animals Prepared/Processed,United States,5.32,0.50,10.64,Avoid,21.17,NaN,0.24,Tyson Fire Stingers Fully Cooked Breaded Bone-...
107379,2650314,Kellogg Company US,Morningstar Farms,"INGREDIENTS: WATER, SOYBEAN OIL, MODIFIED CORN...",16.00,GRM,Vegetable Based Products / Meals,United States,12.60,0.50,25.20,Avoid,11.40,NaN,0.60,Morningstar Farms Breakfast Bacon 5.25oz


In [14]:
# Get the number of unique values in a specific column
unique_brand_cat_count = df_food['branded_food_category'].nunique()
print(f"Seems like we have '{unique_brand_cat_count.compute()}' unique brand categories.")

Seems like we have '293' unique brand categories.


In [10]:
# Get the number of unique values in a specific column
unique_brand_owner_count = df_food['brand_name'].nunique()
print(f"Seems like we have '{unique_brand_owner_count.compute()}' unique brand owners.")

Seems like we have '26074' unique brand owners.


In [11]:
len(df_food.compute())

1177216

In [ ]:
df_food['split_words'] = df_food['branded_food_category'].str.split(',')
flattened = df_food.explode('split_words')
unique_words = flattened['split_words'].unique()
unique_words_result = unique_words.compute()
print(unique_words_result)
unique_words_df = pd.DataFrame({'Unique Words': unique_words_result})
unique_words_df.to_csv("unique_branded_food_categories.csv", index=False)

0                       Pre-Packaged Fruit & Vegetables
1                                          Dips & Salsa
2                                  Mexican Dinner Mixes
3                               Marinades & Tenderizers
4                             Yogurt/Yogurt Substitutes
                            ...                        
36     meant to simulate the taste and mouthfeel of ...
37                   Pork Sausages - Prepared/Processed
38    Includes any products that can be described/ob...
39                                                wheat
40                                                 nuts
Name: split_words, Length: 439, dtype: object


In [55]:
unique_values = df_food['Category'].unique().compute()
result = {}
for value in unique_values:
    filtered_dd = df_food[df_food['Category'] == value] 
    sorted_dd = filtered_dd.sort_values(by='Ratio', ascending=True)
    #count = len(filtered_dd)
    #avg_ratio = filtered_dd['Ratio'].mean().compute()
    #info = {}
    #info["count"] = count
    #info["average_ratio"] = avg_ratio.round(2)
    #result[value] = info
    new_filtered_df = sorted_dd.compute()
    filename = value + '.csv'
    new_filtered_df.to_csv(filename, index=True)


In [44]:
import json

# Saving the dictionary as a JSON file
with open("ratiocount1.json", "w") as json_file:
    json.dump(result, json_file, indent=4)  # The 'indent' parameter makes the file more readable

print("Dictionary has been saved as my_data.json!")

Dictionary has been saved as my_data.json!


In [ ]:
category_keywords = {
    "Fruits": ["fruit", "fruits"],
    "Vegetables": ["leafy", "greens", "vegetable", "legume", "legumes", "vegetables", "green"],
    "Meat and Pultry": ["meat", "meats", "poultry"],
    "Fish and Seafood": ["fish", "seafood"],
    "Grains": ["rice", "wheat", "oats", "corn", "grain", "grains", "bread"],
    "Dairy": ["milk", "cheese", "yogurt"],
    "Eggs": ["egg", "eggs"],
    "Nuts and Seeds": ["peanut", "peanuts", "almonds"],
    "Oils and Fats": ["oil", "olive"],
    "Sweets and Snacks": ["candy", "candies", "chips", "baked", "dessert", "desserts", ""]

    # Add more categories and their associated keywords as needed
}

In [54]:
# Function to check if a category keyword is in the brand_food_category
def categorize_food(brand_food_category: str) -> str:
    for category, keywords in category_keywords.items():
        for keyword in keywords:
            if keyword.lower() in brand_food_category.lower():
                return category
    return "uncategorized"  # Default category if no match is found

# Function to apply categorization to the dataframe
def apply_category(df):
    df['food_category'] = df['branded_food_category'].apply(categorize_food, meta=('x', 'str'))
    return df

<h1>BEST</h1>

In [69]:
df_best_food = dd.read_csv("Best.csv", dtype=dtype_dict)
df_ideal_food = dd.read_csv("Ideal.csv", dtype=dtype_dict)
df_moderate_food = dd.read_csv("Moderate.csv", dtype=dtype_dict)
df_avoid_food = dd.read_csv("Avoid.csv", dtype=dtype_dict)

In [57]:
top_best = df_best_food.head(5)
top_good = df_ideal_food.head(5)
top_poor = df_moderate_food.head(5)
top_worst = df_avoid_food.head(5)


In [70]:
best_top5 = df_best_food[df_best_food['Fiber'] > 1].head(5)
ideal_top5 = df_ideal_food[df_ideal_food['Fiber'] > 1].head(5)
moderate_top5 = df_moderate_food[df_moderate_food['Fiber'] > 1].head(5)
avoid_top5 = df_avoid_food[df_avoid_food['Fiber'] > 1].head(5)


In [71]:
best_20 = dd.concat([best_top5, ideal_top5, moderate_top5, avoid_top5])

In [58]:
top20_df = dd.concat([top_best, top_good, top_poor, top_worst])

In [65]:
computed_top20 = top20_df.compute()

In [72]:
computed_best_20 = best_20.compute()

In [66]:
computed_top20

,Unnamed: 0,fdc_id,brand_owner,brand_name,ingredients,serving_size,serving_size_unit,branded_food_category,market_country,Carbs,Fiber,Ratio,Category,Proteins,Total_Fat,Total_Sugars,description
0,94127,2063588,Oleificio Cinquina Srl,CINQUINA,"OLIVES, WATER, SALT, GARLIC, CHILI PEPPERS, WI...",30.00,g,"Pickles, Olives, Peppers & Relishes",United States,0.00,3.30,0.00,Best,0.00,16.67,0.00,GREEN CALABRESE OLIVES
1,103670,2340335,"Wal-Mart Stores, Inc.",FRESHNESS GUARANTEED,"CREAM CHEESE (MILK, CREAM, SALT, CAROB BEAN GU...",235.00,g,"Cakes, Cupcakes, Snack Cakes",United States,0.00,0.90,0.00,Best,4.68,22.55,21.28,"CARAMEL TOPPED NEW YORK STYLE CHEESECAKE, CARA..."
2,45949,2479406,"Suja Life, LLC",SUJA,"PURIFIED SPARKLING WATER, ORGANIC ORANGE JUICE...",354.00,ml,"Fruit & Vegetable Juice, Nectars & Fruit Drinks",United States,0.00,0.30,0.00,Best,0.00,0.00,2.26,GINGER CITRUS COLD-PRESSED SPARKLING FRUIT JUI...
3,103883,2450291,"Meijer, Inc.",MEIJER,"MUSHROOMS, WATER, SALT, AND CALCIUM DISODIUM E...",113.00,g,Canned Vegetables,United States,0.00,2.70,0.00,Best,2.65,0.00,0.00,MUSHROOM STEMS & PIECES
4,45282,2378662,"Innoventions, Inc.",SUSHI SAM,"COOKED RICE, VINEGAR, SHRIMP TEMPURA (SEASONED...",227.00,g,Sushi,United States,0.00,1.30,0.00,Best,4.85,1.32,0.44,CRUNCHY SHRIMP TEMPURA
0,75317,1277123,Blue Diamond Growers,BLUE DIAMOND ALMONDS,"ALMONDS, SUGAR, COCOA POWDER (PROCESSED WITH A...",28.00,g,"Popcorn, Peanuts, Seeds & Related Snacks",United States,32.14,10.70,3.00,Ideal,17.86,46.43,14.29,"SEA SALT DARK CHOCOLATE ALMONDS, SEA SALT DARK..."
1,106031,1132714,"JGF Enterprises, LLC",No Brand,"PECANS, PUMPKIN SEEDS, SWEETENED DRIED CRANBER...",18.00,g,"Popcorn, Peanuts, Seeds & Related Snacks",United States,33.33,11.10,3.00,Ideal,11.11,44.44,22.22,"CINNAMON COCONUT SUPERFOOD MIX, CINNAMON COCONUT"
2,7629,1846544,Harris-Teeter Inc.,HARRIS TEETER,ENGLISH PEAS.,85.00,g,Pre-Packaged Fruit & Vegetables,United States,14.12,4.70,3.00,Ideal,5.88,0.00,4.71,ENGLISH PEAS
3,73582,1592833,Nabisco Food Company,PLANTERS,"INGREDIENTS: ALMONDS, SUGAR, HONEY, CORN SYRUP...",28.00,g,"Popcorn, Peanuts, Seeds & Related Snacks",United States,32.14,10.70,3.00,Ideal,17.86,42.86,17.86,"PUMPKIN SPICE ALMONDS, PUMPKIN SPICE"
4,33494,1394723,KARMA NUTS INC,KARMA,"CASHEWS, ONION, SEA SALT, LIME POWDER, NATURAL...",28.00,g,"Popcorn, Peanuts, Seeds & Related Snacks",United States,32.14,10.70,3.00,Ideal,14.29,42.86,3.57,"LIME TWIST WRAPPED CASHEWS, LIME TWIST"


In [67]:
columns_to_remove = ['Unnamed: 0']
computed_top20 = computed_top20.drop(columns=columns_to_remove)

In [74]:

# Check for missing values in a specific column (e.g., 'preparation_state_code')
missing_values_count = computed_best_20.isnull().sum()

# Compute and print the number of missing values
print('Missing Values')
print(missing_values_count)

Missing Values
Unnamed: 0               0
fdc_id                   0
brand_owner              0
brand_name               0
ingredients              0
serving_size             0
serving_size_unit        0
branded_food_category    0
market_country           0
Carbs                    0
Fiber                    0
Ratio                    0
Category                 0
Proteins                 0
Total_Fat                0
Total_Sugars             0
description              0
dtype: int64


In [34]:
computed_top20 = computed_top20['brand_name'].fillna("Unknown")

In [75]:
computed_best_20.to_csv('Top20ByRatio1.csv', index=True)

In [64]:
computed_top20.to_csv('Top20ByRatio.csv', index=True)

In [22]:
# Get the number of unique values in a specific column
unique_brand_cat_count_best = df_best_food['branded_food_category'].nunique()
print(f"Seems like we have '{unique_brand_cat_count_best.compute()}' unique brand categories.")

Seems like we have '209' unique brand categories.
